# 03 BiLSTM with fastText embeddings (E4)

`python -m src.embeddings` caches fastText vectors; `python -m src.train_bilstm` trains random / frozen / fine-tuned embeddings with 3 seeds each.

**Architecture:** tokens -> 300-d embedding -> BiLSTM (2 x 128) -> max-pool over time -> dropout 0.3 -> linear -> sigmoid.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('../.cache/mpl'))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 200)
from src import config
S = pd.read_csv(config.RESULTS / 'experiments.csv')          # mean (and sd) over seeds
BY_SEED = pd.read_csv(config.RESULTS / 'experiments_by_seed.csv')

def table(models, sets, variant='clean', split='template', metric='f1'):
    t = S[S.model.isin(models) & S.set.isin(sets) & (S.variant == variant) & (S.split == split)]
    return t.pivot_table(index='model', columns='set', values=metric).reindex(models).round(3)

In [2]:
import torch
from src.train_bilstm import BiLSTM
m = BiLSTM(torch.zeros(5000, 300), trainable=True)
print(m)
print('trainable parameters without the embedding:', sum(p.numel() for n, p in m.named_parameters() if not n.startswith('emb')))

BiLSTM(
  (emb): Embedding(5000, 300, padding_idx=0)
  (lstm): LSTM(300, 128, batch_first=True, bidirectional=True)
  (drop): Dropout(p=0.3, inplace=False)
  (out): Linear(in_features=256, out_features=1, bias=True)
)
trainable parameters without the embedding: 440577


## Results over 3 seeds

In [3]:
models = ['bilstm_random', 'bilstm_frozen', 'bilstm_finetuned']
t = S[S.model.isin(models) & (S.variant == 'clean') & S.set.isin(['val', 'test', 'test/tpl', 'chichewa/all'])]
t.pivot_table(index='model', columns='set', values=['f1', 'f1_sd']).round(3)

f1                  ...  f1_sd                
set              chichewa/all   test test/tpl  ...   test test/tpl    val
model                                          ...                       
bilstm_finetuned        0.738  0.860    0.980  ...  0.107    0.017  0.004
bilstm_frozen           0.733  0.778    0.983  ...  0.018    0.006  0.000
bilstm_random           0.597  0.791    0.990  ...  0.027    0.000  0.000

[3 rows x 8 columns]

In [4]:
BY_SEED[BY_SEED.model.isin(models) & (BY_SEED.variant == 'clean') & (BY_SEED.set == 'test')][['model', 'seed', 'precision', 'recall', 'f1', 'pr_auc']].round(3)

,model,seed,precision,recall,f1,pr_auc
931,bilstm_finetuned,13,1.000,0.617,0.763,0.994
1000,bilstm_finetuned,42,0.987,0.963,0.975,0.994
1069,bilstm_finetuned,2026,0.953,0.753,0.841,0.983
1651,bilstm_frozen,13,0.980,0.617,0.758,0.976
1720,bilstm_frozen,42,0.982,0.654,0.785,0.982
1789,bilstm_frozen,2026,1.000,0.654,0.791,0.985
1858,bilstm_random,13,1.000,0.691,0.818,1.000
1927,bilstm_random,42,1.000,0.617,0.763,0.914
1996,bilstm_random,2026,1.000,0.654,0.791,0.985


**Reading.** Validation F1 is 0.99-1.00 for every variant, yet test F1 is 0.78-0.86: the test set's largest template (31 landlord-impersonation scams with no phone number) is caught by some seeds and missed by others (fine-tuned vectors: 0%, 35% and 90% of it), hence the large spread. On one message per template all variants score 0.98-0.99. On Chichewa, pretrained vectors raise F1 (0.73-0.74 vs 0.60 from random init), but random init ranks better (PR-AUC 0.80 vs 0.65-0.70): the F1 gain is partly a calibration effect at the 0.5 threshold, not clearly better transfer.

In [5]:
from src.embeddings import load_cache
import numpy as np
data = np.load(config.CACHE / 'fasttext_tokens.npz')
print(f"{len(data['tokens'])} tokens cached, {data['in_vocab'].mean():.1%} are whole words in fastText's vocabulary; the rest get sub-word vectors")

4559 tokens cached, 39.2% are whole words in fastText's vocabulary; the rest get sub-word vectors
